<h2><a href="https://leetcode.com/problems/n-queens/">51. N-Queens</a></h2><h3>Hard</h3><hr><p>The <strong>n-queens</strong> puzzle is the problem of placing <code>n</code> queens on an <code>n x n</code> chessboard such that no two queens attack each other.</p>

<p>Given an integer <code>n</code>, return <em>all distinct solutions to the <strong>n-queens puzzle</strong></em>. You may return the answer in <strong>any order</strong>.</p>

<p>Each solution contains a distinct board configuration of the n-queens&#39; placement, where <code>&#39;Q&#39;</code> and <code>&#39;.&#39;</code> both indicate a queen and an empty space, respectively.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>
<img alt="" src="https://assets.leetcode.com/uploads/2020/11/13/queens.jpg" style="width: 600px; height: 268px;" />
<pre>
<strong>Input:</strong> n = 4
<strong>Output:</strong> [[&quot;.Q..&quot;,&quot;...Q&quot;,&quot;Q...&quot;,&quot;..Q.&quot;],[&quot;..Q.&quot;,&quot;Q...&quot;,&quot;...Q&quot;,&quot;.Q..&quot;]]
<strong>Explanation:</strong> There exist two distinct solutions to the 4-queens puzzle as shown above
</pre>

<p><strong class="example">Example 2:</strong></p>

<pre>
<strong>Input:</strong> n = 1
<strong>Output:</strong> [[&quot;Q&quot;]]
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>1 &lt;= n &lt;= 9</code></li>
</ul>


## Intuition & Approach

- The n‑queens puzzle asks us to place one queen in each column of an `n×n` board such that **no two queens attack** each other. In chess, a queen attacks along its row, column, and both diagonals.
- We can build a solution **column by column**. When placing a queen in column `c`, all earlier columns `0..c-1` are already filled with non‑attacking queens. We only need to ensure the new queen does not conflict with those.
- **Backtracking** is the natural strategy: try every row in the current column, check if it is safe, place a queen, recurse to the next column, and undo the placement when returning (i.e., backtrack).
- A 1‑D list of strings models the board. Each string has length `n`; a `'Q'` marks a queen and `'.'` marks an empty square.

## Code Walkthrough (Brute Force with Scanning)

**Main function:**
```python
def solveNQueens(self, n: int) -> List[List[str]]:
    ans = []                              # store all valid boards
    board = ["." * n for _ in range(n)]  # start with an empty board
    self.solve(board, 0, ans, n)          # begin with column 0
    return ans
```
- `ans` accumulates final configurations.
- `board` is a list of n strings, each of length n. We update rows by string slicing when placing/removing a queen.
- We start at column 0 and build solutions column by column.

**Recursive backtracking function:**
```python
def solve(self, board, col, ans, n):
    if col == n:                          # all columns filled successfully
        ans.append(list(board))           # record this solution
        return
    
    for row in range(n):                  # try each row in the current column
        if self.isSafe(row, col, board, n):  # check if placement is safe
            # Place queen
            board[row] = board[row][:col] + "Q" + board[row][col+1:]
            self.solve(board, col+1, ans, n)  # move to next column
            # Backtrack: remove queen
            board[row] = board[row][:col] + "." + board[row][col+1:]
```

**Key idea in each step:**
- When `col == n`, we've placed n queens in n columns successfully—add this solution to results.
- For each column, we iterate through all possible rows.
- **String slicing trick:** `board[row][:col] + "Q" + board[row][col+1:]` replaces position `col` with a queen without modifying adjacent characters.

**Safety check function:**
```python
def isSafe(self, row, col, board, n):
    duprow, dupcol = row, col  # save original position
    
    # DIRECTION 1: Check upper-left diagonal (row decreases, col decreases)
    while row >= 0 and col >= 0:
        if board[row][col] == "Q":
            return False
        row -= 1
        col -= 1
    
    # DIRECTION 2: Check left side of the same row (col decreases)
    col = dupcol
    row = duprow
    while col >= 0:
        if board[row][col] == "Q":
            return False
        col -= 1
    
    # DIRECTION 3: Check lower-left diagonal (row increases, col decreases)
    row = duprow
    col = dupcol
    while row < n and col >= 0:
        if board[row][col] == "Q":
            return False
        row += 1
        col -= 1
    
    return True
```

**Why these three directions?**
- Since we place queens **from left to right** (column by column), all previously placed queens are to the **left** of the current column.
- We only need to check three directions: upper-left diagonal, left row, and lower-left diagonal.
- We don't check right, upper-right, or lower-right because those columns are still empty.

## 🔍 Detailed Dry Run (n = 4)

```
Initial state: board = ["....", "....", "....", "...."]

col=0:
  row=0: isSafe(0,0) → True (empty board)
    Place Q: ["Q...", "....", "....", "...."]
    
    col=1:
      row=0: isSafe(0,1) → False (left direction has Q at (0,0))
      row=1: isSafe(1,1) → False (upper-left diagonal has Q at (0,0))
      row=2: isSafe(2,1) → True
        Place Q: ["Q...", "....", ".Q..", "...."]
        
        col=2:
          row=0: isSafe(0,2) → False (left has Q)
          row=1: isSafe(1,2) → False (upper-left has Q at (2,1))
          row=2: isSafe(2,2) → False (left has Q)
          row=3: isSafe(3,2) → False (upper-left diagonal blocked)
        Backtrack: Remove Q from (2,1)
        
      row=3: isSafe(3,1) → True
        Place Q: ["Q...", "....", "....", "...Q"]
        
        col=2:
          row=1: isSafe(1,2) → True
            Place Q: ["Q...", ".Q..", "....", "...Q"]
            
            col=3:
              row=3: isSafe(3,3) → True
                Place Q: ["Q...", ".Q..", "....", "..QQ"]
                → INVALID! (same row)  (CONTINUED RECURSION)
              
              (After trying all rows in col=3, backtrack)
            Backtrack: Remove Q from (1,2)
          ...
```

This continues until all combinations are explored. Two valid complete configurations exist for n=4.

## Edge Cases

- **n = 1:** Returns `[['Q']]` ✓ (single queen on 1×1 board)
- **n = 0:** Problem constraints guarantee `1 ≤ n ≤ 9`, so this is outside spec. Current code would return `[[]]` (empty solution), which may or may not be intended.
- **Negative n:** Also outside constraints. The code would not execute properly (list comprehension would produce empty list).
- **n = 2 or 3:** No valid solutions exist. The code correctly returns an empty `ans = []`.

## Time Complexity Analysis

| Case | Complexity | Explanation |
|------|-----------|-------------|
| **Worst case** | **O(N!)** | Every column has up to N rows to try; recursion explores N × (N-1) × (N-2) × ... × 1 ≈ N! branches. Each `isSafe` call is O(N). Total: O(N! × N). |
| **Best case** | **O(N!)** | Even with early termination, the algorithm searches the full recursion tree in most scenarios. No solutions exist for n = 2,3, so `O(N!)` still applies. |
| **Average case** | **O(N!)** | Similar to worst case—backtracking explores a large portion of the tree. |

**Why O(N!) and not less?**
- The number of valid N‑queens configurations grows as a permutation: there's no simple formula, but empirically it's proportional to N!.
- Each `isSafe` call scans up to 3 directions, each up to O(N), contributing a factor of O(N).

## Space Complexity Analysis

| Component | Complexity | Explanation |
|-----------|-----------|-------------|
| **Board representation** | O(N²) | A list of N strings, each of length N. |
| **Recursion call stack** | O(N) | Maximum depth is N (one level per column). |
| **Result storage (ans)** | O(K × N²) | K = number of solutions (~N! / large factorial factor). Each solution is a copy of the N² board. |
| **Total** | **O(K × N²)** | Dominated by storing all solutions. For N=8, K≈92, so factor is manageable. |

---

### Optimized Approach (Using Hashing with Tracks)

The brute-force version rescans the entire board in `isSafe`, which is slower for large N. The optimized version maintains **three hash sets** to track occupied rows and diagonals:

```python
def solveNQueens(self, n: int) -> List[List[str]]:
    ans = []
    board = ["." * n for _ in range(n)]
    leftrow = [0] * n              # track which rows have queens
    lowerDiagonal = [0] * (2*n-1)  # track lower-left to upper-right diagonals
    upperDiagonal = [0] * (2*n-1)  # track upper-left to lower-right diagonals
    self.solve(board, 0, ans, leftrow, upperDiagonal, lowerDiagonal, n)
    return ans

def solve(self, board, col, ans, leftrow, upperDiagonal, lowerDiagonal, n):
    if col == n:
        ans.append(list(board))
        return
    
    for row in range(n):
        # Check if this row and both diagonals are free
        if (leftrow[row] == 0 
            and lowerDiagonal[row + col] == 0 
            and upperDiagonal[n - 1 + col - row] == 0):
            
            # Place queen
            board[row] = board[row][:col] + "Q" + board[row][col+1:]
            leftrow[row] = 1
            lowerDiagonal[row + col] = 1
            upperDiagonal[n - 1 + col - row] = 1
            
            # Recurse
            self.solve(board, col+1, ans, leftrow, upperDiagonal, lowerDiagonal, n)
            
            # Backtrack
            board[row] = board[row][:col] + "." + board[row][col+1:]
            leftrow[row] = 0
            lowerDiagonal[row + col] = 0
            upperDiagonal[n - 1 + col - row] = 0
```

**Key improvements:**
1. **Constant-time checks:** Instead of scanning 3 directions (O(N)), we check arrays in O(1).
2. **Diagonal indexing:**
   - `lowerDiagonal[row + col]`: uniquely identifies each lower-left to upper-right diagonal.
   - `upperDiagonal[n - 1 + col - row]`: uniquely identifies each upper-left to lower-right diagonal.

**Time complexity with optimization:** O(N! × 1) = **O(N!)** for the tree exploration, but with a much smaller constant factor.

**Space complexity:** Same O(K × N²) for results, but temporary arrays add O(N), which is negligible.



In [9]:
from typing import List
class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:
        ans = []
        board = ["." * n for _ in range(n)]
        self.solve(board,0,ans,n)
        return ans
    
    def solve(self,board,col,ans,n):
        if col == n:
            ans.append(list(board))
            return
        
        for row in range(n):
            if self.isSafe(row,col,board,n):
                board[row] = board[row][:col]+"Q"+board[row][col+1:]
                self.solve(board,col+1,ans,n)
                board[row] = board[row][:col]+"."+board[row][col+1:]
    
    def isSafe(self,row,col,board,n):
        duprow = row
        dupcol = col

        while row >=0 and col>=0:
            if board[row][col] == "Q":
                return False
            row -= 1
            col -= 1

        col = dupcol
        row = duprow
        while col >= 0:
            if board[row][col] == "Q":
                return False
            col -= 1
        
        row = duprow
        col = dupcol
        while row < n and col >= 0:
            if board[row][col] == "Q":
                return False
            row+= 1
            col -= 1
        return True


n = 4
print(Solution().solveNQueens(n))


[['..Q.', 'Q...', '...Q', '.Q..'], ['.Q..', '...Q', 'Q...', '..Q.']]


In [10]:
from typing import List
class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:
        ans = []
        board = ["." * n for _ in range(n)]
        leftrow = [0] * n
        lowerDiagonal = [0] * (2 * n -1)
        upperDiagonal = [0] * (2 * n - 1)
        self.solve(board,0,ans,leftrow,upperDiagonal,lowerDiagonal,n)
        return ans
    
    def solve(self,board,col,ans,leftrow,upperDiagonal,lowerDiagonal,n):
        if col == n:
            ans.append(list(board))
            return
        
        for row in range(n):
            if(leftrow[row] == 0
               and lowerDiagonal[row+col] == 0
               and upperDiagonal[n-1+col-row] == 0
            ):
                board[row] = board[row][:col] + "Q" + board[row][col+1:]
                leftrow[row] = 1
                lowerDiagonal[row + col] = 1
                upperDiagonal[n-1 + col - row] = 1
                self.solve(board,col+1,ans,leftrow,upperDiagonal,lowerDiagonal,n)
                board[row] = board[row][:col] + "." + board[row][col+1:]
                leftrow[row] = 0
                lowerDiagonal[row + col] = 0
                upperDiagonal[n-1 + col - row] = 0


n = 4
print(Solution().solveNQueens(n))

[['..Q.', 'Q...', '...Q', '.Q..'], ['.Q..', '...Q', 'Q...', '..Q.']]
